# 5. Linear Regression: VIF, Residual Plots, Cook's Distance
Works on **any dataset with a numeric target**. A **file explorer window** opens in Step 2, then you type the target column name.

**Steps:** Pick file -> Prepare data -> VIF (multicollinearity) -> Fit OLS regression -> Residual diagnostics -> Cook's distance (influential points) -> Test-set performance

> VIF, OLS, residual diagnostics and Cook's distance are all computed with **numpy/scipy** (no extra libraries needed). The formulas are visible in the code, which helps in a viva.

## Step 0: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## Step 1: CONFIG (defaults are fine)

In [ ]:
ALPHA            = 0.05      # significance level
VIF_THRESHOLD    = 10        # features with VIF above this are removed one by one (5 = stricter, 10 = common)
REMOVE_HIGH_VIF  = True      # False = only show VIF, do not remove anything
TEST_SIZE        = 0.2       # for the train/test evaluation (Step 9)
RANDOM_STATE     = 42
MAX_CATEGORIES   = 15        # categorical columns with more levels than this are dropped
DROP_COLS        = []        # columns to drop manually, e.g. ['id', 'name']

## Step 2: Select Dataset from File Explorer

In [ ]:
def pick_file():
    """Opens the OS file explorer. Falls back to typing the path if no GUI is available."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        path = filedialog.askopenfilename(
            title='Select your dataset',
            filetypes=[('Data files', '*.csv *.xlsx *.xls *.json *.txt *.tsv'), ('All files', '*.*')])
        root.destroy()
    except Exception:
        print('File dialog not available -> type the path instead.')
        path = input('Enter full file path: ')
    return path.strip().strip('"').strip("'")

def load_data(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    if ext == '.json':
        return pd.read_json(path)
    try:
        return pd.read_csv(path, sep=None, engine='python')
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=None, engine='python', encoding='latin-1')

file_path = pick_file()
if not file_path:
    raise SystemExit('No file selected. Re-run this cell.')

df = load_data(file_path)
df.columns = df.columns.astype(str).str.strip()
print('Loaded:', file_path, '| Shape:', df.shape)
df.head()

In [ ]:
def text_cols(d):
    return [col for col in d.columns
            if pd.api.types.is_object_dtype(d[col]) or pd.api.types.is_string_dtype(d[col])]

# convert numbers stored as text, so numeric columns are recognised correctly
for col in text_cols(df):
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())
df = df.replace(['?', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'null', 'NULL', 'None', '-', '--', ''], np.nan)
for col in text_cols(df):
    conv = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() > 0 and conv.notna().sum() >= 0.9 * df[col].notna().sum():
        df[col] = conv

num_list = df.select_dtypes(include=np.number).columns.tolist()
print('Numeric columns (possible targets):', num_list)
TARGET = input('Enter the numeric TARGET column to predict: ').strip()
while TARGET not in num_list:
    TARGET = input(f"'{TARGET}' is not a numeric column. Enter a valid numeric column: ").strip()
print('TARGET =', TARGET)
if df[TARGET].nunique() <= 10:
    print('WARNING: target has <= 10 unique values - this may be a classification problem.')

## Step 3: Prepare Data (missing values, encoding)

In [ ]:
# NOTE: row labels (index) of the original file are kept, so influential rows can be traced back to your data.
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET])
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(str)

# drop useless columns
to_drop = [col for col in DROP_COLS if col in df.columns]
for col in df.columns:
    if col == TARGET:
        continue
    if (df[col].isnull().mean() > 0.5 or df[col].nunique(dropna=True) <= 1
            or (col in text_cols(df) and df[col].nunique() == len(df))
            or (col in text_cols(df) and df[col].nunique() > MAX_CATEGORIES)
            or pd.api.types.is_datetime64_any_dtype(df[col])):
        to_drop.append(col)
to_drop = list(dict.fromkeys(to_drop))
print('Dropped columns:', to_drop if to_drop else 'None')
df = df.drop(columns=to_drop)

y = df[TARGET].astype(float)
X = df.drop(columns=TARGET)
num_c = X.select_dtypes(include=np.number).columns.tolist()
cat_c = [col for col in X.columns if col not in num_c]

# impute: median (numeric), mode (categorical)
for col in num_c:
    X[col] = X[col].fillna(X[col].median())
for col in cat_c:
    X[col] = X[col].fillna(X[col].mode().iloc[0])

# one-hot encode categoricals (drop_first avoids the dummy variable trap)
X = pd.get_dummies(X, columns=cat_c, drop_first=True, dtype=float).astype(float)
X = X.loc[:, X.nunique() > 1]                      # remove constant columns

print('Numeric features    :', num_c)
print('Categorical features:', cat_c, '(one-hot encoded)')
print(f'Final: n = {len(X)} observations, {X.shape[1]} predictors')
if len(X) < 10 * X.shape[1]:
    print('WARNING: fewer than 10 observations per predictor - results may be unstable.')
X.head()

## Step 4: Quick Look at Target and Relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, kde=True, ax=axes[0], color='steelblue'); axes[0].set_title(f'Distribution of {TARGET} (skew = {y.skew():.2f})')
corr_t = X.apply(lambda s: s.corr(y)).sort_values(key=abs, ascending=False)
top = corr_t.head(15)
axes[1].barh(top.index[::-1], top.values[::-1], color=['tomato' if v < 0 else 'seagreen' for v in top.values[::-1]])
axes[1].set_title(f'Correlation of predictors with {TARGET}'); axes[1].axvline(0, color='k', lw=0.8)
plt.tight_layout(); plt.show()

# scatter plots of the 4 most correlated predictors
cont_top = [f for f in corr_t.index if X[f].nunique() > 2][:4]
if cont_top:
    fig, axes = plt.subplots(1, len(cont_top), figsize=(4.5 * len(cont_top), 4), squeeze=False)
    for ax, f in zip(axes[0], cont_top):
        sns.regplot(x=X[f], y=y, ax=ax, scatter_kws={'alpha': 0.4, 's': 15}, line_kws={'color': 'red'})
        ax.set_title(f'{f}  (r = {corr_t[f]:.2f})')
    plt.tight_layout(); plt.show()

## Step 5: VIF - Variance Inflation Factor (Multicollinearity)
$VIF_j = \dfrac{1}{1 - R_j^2}$ where $R_j^2$ comes from regressing predictor $j$ on **all the other predictors**.

| VIF | Meaning |
|---|---|
| 1 | No collinearity |
| 1 - 5 | Low / acceptable |
| 5 - 10 | Moderate (watch out) |
| > 10 | Serious multicollinearity -> unstable, hard-to-interpret coefficients |

In [ ]:
def compute_vif(Xd):
    A = Xd.values.astype(float)
    out = {}
    for j, col in enumerate(Xd.columns):
        yj = A[:, j]
        Z = np.column_stack([np.ones(len(A)), np.delete(A, j, axis=1)])   # other predictors + intercept
        beta = np.linalg.lstsq(Z, yj, rcond=None)[0]
        ss_res = ((yj - Z @ beta) ** 2).sum()
        ss_tot = ((yj - yj.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        out[col] = np.inf if r2 >= 1 - 1e-12 else 1 / (1 - r2)
    return pd.Series(out, name='VIF')

vif0 = compute_vif(X).sort_values(ascending=False)
vif_table = pd.DataFrame({'VIF': vif0, 'Tolerance (1/VIF)': 1 / vif0,
                          'Level': pd.cut(vif0, [0, 5, 10, np.inf], labels=['OK (<5)', 'Moderate (5-10)', 'HIGH (>10)'])})
display(vif_table)

plt.figure(figsize=(8, max(3, 0.35 * len(vif0))))
plt.barh(vif0.index[::-1], vif0.clip(upper=50).values[::-1],
         color=['tomato' if v > 10 else 'orange' if v > 5 else 'seagreen' for v in vif0.values[::-1]])
plt.axvline(5, color='orange', ls='--', label='5'); plt.axvline(10, color='red', ls='--', label='10')
plt.xlabel('VIF (capped at 50 for display)'); plt.title('Variance Inflation Factors'); plt.legend()
plt.show()

In [ ]:
# Remove the feature with the highest VIF, recompute, repeat until all VIF <= threshold
feats = list(X.columns)
removed = []
if REMOVE_HIGH_VIF:
    while len(feats) > 1:
        v = compute_vif(X[feats])
        worst = v.idxmax()
        if v[worst] > VIF_THRESHOLD:
            print(f'Removing {worst!r}  (VIF = {v[worst]:.2f})')
            removed.append(worst)
            feats.remove(worst)
        else:
            break
    print('\nRemoved because of multicollinearity:', removed if removed else 'None')
else:
    print('REMOVE_HIGH_VIF = False -> all features kept')

X_final = X[feats]
print('\nVIF after removal:')
display(compute_vif(X_final).sort_values(ascending=False).to_frame())

## Step 6: Fit the OLS Linear Regression
$y = \beta_0 + \beta_1x_1 + \dots + \beta_px_p + \varepsilon$, estimated by least squares: $\hat{\beta} = (X^TX)^{-1}X^Ty$

In [ ]:
def fit_ols(Xd, yv):
    Xc = np.column_stack([np.ones(len(Xd)), Xd.values.astype(float)])     # add intercept column
    yy = yv.values.astype(float)
    n, p = Xc.shape
    XtX_inv = np.linalg.pinv(Xc.T @ Xc)
    beta = XtX_inv @ Xc.T @ yy
    fitted = Xc @ beta
    resid = yy - fitted
    sse = resid @ resid
    sst = ((yy - yy.mean()) ** 2).sum()
    dof = n - p
    mse = sse / dof
    se = np.sqrt(np.diag(XtX_inv) * mse)
    t = beta / se
    pval = 2 * stats.t.sf(np.abs(t), dof)
    tc = stats.t.ppf(1 - ALPHA / 2, dof)
    r2 = 1 - sse / sst
    adj = 1 - (1 - r2) * (n - 1) / dof
    F = ((sst - sse) / (p - 1)) / mse if p > 1 else np.nan
    pF = stats.f.sf(F, p - 1, dof) if p > 1 else np.nan
    llf = -n / 2 * (np.log(2 * np.pi) + np.log(sse / n) + 1)
    h = np.einsum('ij,jk,ik->i', Xc, XtX_inv, Xc)                          # leverage (diagonal of hat matrix)
    std_resid = resid / np.sqrt(mse * (1 - h))                              # internally standardised residuals
    cooks = std_resid ** 2 * h / (p * (1 - h))                              # Cook's distance
    names = ['Intercept'] + list(Xd.columns)
    coef = pd.DataFrame({'coef': beta, 'std err': se, 't': t, 'p-value': pval,
                         f'CI low': beta - tc * se, f'CI high': beta + tc * se}, index=names)
    coef['sig'] = np.select([pval < 0.001, pval < 0.01, pval < 0.05, pval < 0.1], ['***', '**', '*', '.'], '')
    sd_x, sd_y = Xd.std(ddof=1).values, yv.std(ddof=1)
    coef['std. beta'] = [np.nan] + list(beta[1:] * sd_x / sd_y)
    return dict(coef=coef, n=n, p=p, dof=dof, r2=r2, adj=adj, F=F, pF=pF, mse=mse, rmse=np.sqrt(mse),
                aic=-2 * llf + 2 * p, bic=-2 * llf + p * np.log(n), sse=sse, sst=sst, Xc=Xc, XtX_inv=XtX_inv,
                fitted=fitted, resid=resid, h=h, std_resid=std_resid, cooks=cooks, index=Xd.index)

m = fit_ols(X_final, y)

print('=' * 72)
print(f'OLS REGRESSION RESULTS   (target: {TARGET})')
print('=' * 72)
print(f'Observations (n)        : {m["n"]}')
print(f'Predictors (k)          : {m["p"] - 1}')
print(f'R-squared               : {m["r2"]:.4f}')
print(f'Adjusted R-squared      : {m["adj"]:.4f}')
print(f'F-statistic             : {m["F"]:.4f}   (df = {m["p"]-1}, {m["dof"]})')
print(f'Prob (F-statistic)      : {m["pF"]:.3e}')
print(f'Residual std error (RMSE): {m["rmse"]:.4f}')
print(f'AIC / BIC               : {m["aic"]:.2f} / {m["bic"]:.2f}')
print('=' * 72)
display(m['coef'])
print('Significance codes: *** p<0.001, ** p<0.01, * p<0.05, . p<0.1')

In [ ]:
# Interpretation in words
coef = m['coef']
sig = coef[(coef['p-value'] <= ALPHA)].drop(index='Intercept', errors='ignore')
print(f'* The model explains {m["r2"]*100:.1f}% of the variance in {TARGET} (adjusted R-squared = {m["adj"]:.3f}).')
print(f'* Overall F-test: p = {m["pF"]:.3e} -> the model is ' + ('statistically significant (at least one predictor matters).' if m['pF'] <= ALPHA else 'NOT significant.'))
print(f'* Significant predictors (p <= {ALPHA}): {list(sig.index) if len(sig) else "none"}')
for name, row in sig.iterrows():
    direction = 'increases' if row['coef'] > 0 else 'decreases'
    print(f'    - one-unit increase in {name} {direction} {TARGET} by {abs(row["coef"]):.4f} on average (other predictors held constant)')
ns = coef[(coef['p-value'] > ALPHA)].drop(index='Intercept', errors='ignore')
print(f'* Not significant: {list(ns.index) if len(ns) else "none"}')

# Coefficient plot with confidence intervals
cp = coef.drop(index='Intercept')
if len(cp):
    plt.figure(figsize=(8, max(3, 0.4 * len(cp))))
    ypos = np.arange(len(cp))
    plt.errorbar(cp['coef'], ypos, xerr=[cp['coef'] - cp['CI low'], cp['CI high'] - cp['coef']], fmt='o', capsize=3)
    plt.axvline(0, color='red', ls='--'); plt.yticks(ypos, cp.index)
    plt.title(f'Coefficients with {(1-ALPHA)*100:.0f}% CI (CI crossing 0 = not significant)')
    plt.show()

In [ ]:
# (Optional) the same model with statsmodels - prints the classic summary table if the library is installed
try:
    import statsmodels.api as sm
    print(sm.OLS(y, sm.add_constant(X_final)).fit().summary())
except ImportError:
    print('statsmodels is not installed - the table above already contains the same numbers.')

## Step 7: Residual Diagnostics
Residual $e_i = y_i - \hat{y}_i$. A good model has residuals that look like **random noise**.

| Plot | What to look for | Assumption checked |
|---|---|---|
| Residuals vs Fitted | random cloud around 0, flat trend line | Linearity, constant variance |
| Normal Q-Q | points on the diagonal | Normality of errors |
| Scale-Location | flat trend line | Homoscedasticity (equal variance) |
| Residuals vs Leverage | no points far right / far up | Influential observations |

In [ ]:
fitted, resid, sres, lev, cooks = m['fitted'], m['resid'], m['std_resid'], m['h'], m['cooks']
n, p = m['n'], m['p']
bins = pd.qcut(fitted, q=max(2, min(10, n // 20)), duplicates='drop')

def trend(vals):
    g = pd.DataFrame({'f': fitted, 'v': vals}).groupby(bins)
    return g['f'].mean().values, g['v'].mean().values

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, resid, alpha=0.5, s=18)
ax.axhline(0, color='red', ls='--')
ax.plot(*trend(resid), color='orange', lw=2, marker='o', label='binned mean')
ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals'); ax.set_title('Residuals vs Fitted'); ax.legend()

# 2. Normal Q-Q
ax = axes[0, 1]
(osm, osr), (slope, intercept, r) = stats.probplot(sres, dist='norm')
ax.scatter(osm, osr, alpha=0.5, s=18)
ax.plot(osm, slope * osm + intercept, color='red')
ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Standardised residuals'); ax.set_title('Normal Q-Q Plot')

# 3. Scale-Location
ax = axes[1, 0]
sl = np.sqrt(np.abs(sres))
ax.scatter(fitted, sl, alpha=0.5, s=18)
ax.plot(*trend(sl), color='orange', lw=2, marker='o')
ax.set_xlabel('Fitted values'); ax.set_ylabel('sqrt(|standardised residuals|)'); ax.set_title('Scale-Location')

# 4. Residuals vs Leverage (point size ~ Cook's distance)
ax = axes[1, 1]
ax.scatter(lev, sres, s=20 + 600 * cooks / max(cooks.max(), 1e-12), alpha=0.4)
ax.axhline(0, color='grey', ls='--'); ax.axhline(3, color='red', ls=':'); ax.axhline(-3, color='red', ls=':')
ax.axvline(2 * p / n, color='purple', ls='--', label=f'leverage = 2p/n = {2*p/n:.3f}')
for i in np.argsort(cooks)[-3:]:
    ax.annotate(str(m['index'][i]), (lev[i], sres[i]), fontsize=9, color='red')
ax.set_xlabel('Leverage'); ax.set_ylabel('Standardised residuals'); ax.set_title('Residuals vs Leverage (size = Cook\'s D)'); ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
# Residual histogram + residuals against each of the strongest predictors (checks linearity per variable)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(resid, kde=True, stat='density', ax=axes[0], color='steelblue')
xs = np.linspace(resid.min(), resid.max(), 200)
axes[0].plot(xs, stats.norm.pdf(xs, 0, resid.std(ddof=1)), 'r--', label='Normal curve')
axes[0].set_title('Histogram of residuals'); axes[0].legend()
axes[1].scatter(y.values, fitted, alpha=0.5, s=18)
lims = [min(y.min(), fitted.min()), max(y.max(), fitted.max())]
axes[1].plot(lims, lims, 'r--'); axes[1].set_xlabel('Actual'); axes[1].set_ylabel('Predicted'); axes[1].set_title('Actual vs Predicted (full data)')
plt.tight_layout(); plt.show()

cont_feats = [f for f in corr_t.index if f in X_final.columns and X_final[f].nunique() > 2][:6]
if cont_feats:
    nc = min(3, len(cont_feats)); nr = int(np.ceil(len(cont_feats) / nc))
    fig, axes = plt.subplots(nr, nc, figsize=(5 * nc, 4 * nr), squeeze=False)
    for ax, f in zip(axes.ravel(), cont_feats):
        ax.scatter(X_final[f], resid, alpha=0.4, s=15); ax.axhline(0, color='red', ls='--')
        ax.set_xlabel(f); ax.set_ylabel('Residuals'); ax.set_title(f'Residuals vs {f}')
    for ax in axes.ravel()[len(cont_feats):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Formal tests of the regression assumptions
print(f'Mean of residuals = {resid.mean():.6f}  (should be ~0)\n')
rows = []

# Normality of residuals
sw_p = stats.shapiro(resid if n <= 5000 else np.random.default_rng(0).choice(resid, 5000, replace=False)).pvalue
jb_stat, jb_p = stats.jarque_bera(resid)
rows.append(['Shapiro-Wilk', 'Residuals are normal', sw_p, 'Normality OK' if sw_p > ALPHA else 'Normality violated'])
rows.append(['Jarque-Bera', 'Residuals are normal', jb_p, 'Normality OK' if jb_p > ALPHA else 'Normality violated'])

# Homoscedasticity: Breusch-Pagan  (regress e^2 on the predictors, LM = n * R^2)
e2 = resid ** 2
b = np.linalg.lstsq(m['Xc'], e2, rcond=None)[0]
r2_bp = 1 - ((e2 - m['Xc'] @ b) ** 2).sum() / ((e2 - e2.mean()) ** 2).sum()
lm = n * r2_bp
bp_p = stats.chi2.sf(lm, p - 1)
rows.append(['Breusch-Pagan', 'Constant variance (homoscedastic)', bp_p, 'Equal variance OK' if bp_p > ALPHA else 'Heteroscedasticity present'])

display(pd.DataFrame(rows, columns=['Test', 'H0', 'p-value', 'Conclusion']))

# Independence: Durbin-Watson (meaningful mainly if rows are in time order)
dw = np.sum(np.diff(resid) ** 2) / np.sum(resid ** 2)
print(f'Durbin-Watson = {dw:.3f}  (~2 = no autocorrelation, <1.5 positive, >2.5 negative autocorrelation; only meaningful for ordered/time data)')
print(f'Skewness of residuals = {stats.skew(resid):.3f} | Kurtosis (excess) = {stats.kurtosis(resid):.3f}')

## Step 8: Cook's Distance - Influential Observations
Cook's distance measures how much **all fitted values change when observation i is removed**:
$D_i = \dfrac{r_i^2}{p} \cdot \dfrac{h_{ii}}{1-h_{ii}}$ ($r_i$ = standardised residual, $h_{ii}$ = leverage, $p$ = number of parameters).

| Rule of thumb | Meaning |
|---|---|
| $D_i > 4/n$ | Potentially influential (common cut-off) |
| $D_i > 1$ | Highly influential |
| Leverage $h_{ii} > 2p/n$ | High-leverage point (unusual predictor values) |
| $|r_i| > 3$ | Outlier in the response |

In [ ]:
thr_cook = 4 / n
thr_lev  = 2 * p / n
infl = cooks > thr_cook
print(f'n = {n}, p = {p}')
print(f'Cook\'s D threshold 4/n = {thr_cook:.4f} -> {infl.sum()} potentially influential observations ({infl.mean()*100:.1f}%)')
print(f'Observations with Cook\'s D > 1        : {(cooks > 1).sum()}')
print(f'High-leverage points (h > 2p/n = {thr_lev:.3f}): {(lev > thr_lev).sum()}')
print(f'Outliers (|standardised residual| > 3) : {(np.abs(sres) > 3).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
ax.vlines(range(n), 0, cooks, color='steelblue', lw=1)
ax.scatter(np.where(infl)[0], cooks[infl], color='red', s=22, zorder=3)
ax.axhline(thr_cook, color='red', ls='--', label=f'4/n = {thr_cook:.4f}')
if cooks.max() > 0.5:
    ax.axhline(1, color='darkred', ls=':', label='D = 1')
for i in np.argsort(cooks)[-5:]:
    ax.annotate(str(m['index'][i]), (i, cooks[i]), fontsize=9, color='red', xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Observation (position)'); ax.set_ylabel("Cook's distance"); ax.set_title("Cook's Distance Plot"); ax.legend()

axes[1].scatter(lev, cooks, alpha=0.5, s=18)
axes[1].axhline(thr_cook, color='red', ls='--'); axes[1].axvline(thr_lev, color='purple', ls='--')
axes[1].set_xlabel('Leverage'); axes[1].set_ylabel("Cook's distance"); axes[1].set_title("Cook's D vs Leverage")
plt.tight_layout(); plt.show()

In [ ]:
# Top influential observations (row labels = index of your original file)
diag = pd.DataFrame({TARGET: y.values, 'fitted': fitted, 'residual': resid, 'std_resid': sres,
                     'leverage': lev, 'cooks_d': cooks}, index=m['index'])
top_infl = diag.sort_values('cooks_d', ascending=False).head(10)
print('Top 10 observations by Cook\'s distance:')
display(top_infl)
print('\nPredictor values of the 5 most influential rows (compare with the rest of the data):')
display(pd.concat([X_final.loc[top_infl.index[:5]], X_final.describe().loc[['mean']]]).round(3))

In [ ]:
# Refit WITHOUT the influential points to see how much they change the model
keep = ~infl
m2 = fit_ols(X_final[keep], y[keep])
comp = pd.DataFrame({'coef (all data)': m['coef']['coef'], 'coef (without influential)': m2['coef']['coef']})
comp['% change'] = (comp.iloc[:, 1] - comp.iloc[:, 0]) / comp.iloc[:, 0].abs() * 100
comp['p (all)'] = m['coef']['p-value']; comp['p (without)'] = m2['coef']['p-value']
display(comp.round(4))

print(pd.DataFrame({'All data': [m['n'], m['r2'], m['adj'], m['rmse']],
                    'Without influential': [m2['n'], m2['r2'], m2['adj'], m2['rmse']]},
                   index=['n', 'R-squared', 'Adj R-squared', 'RMSE']).round(4))
print('\nIf coefficients/R-squared change a lot, the model depends heavily on a few points. '
      'Do NOT delete them automatically: first check whether they are data errors or genuine (rare) cases.')

## Step 9: Predictive Performance on a Test Set
So far the model was fitted on **all** data to study assumptions. Here we check how well it **predicts unseen data**.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_final, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
lr = LinearRegression().fit(X_tr, y_tr)
pred_tr, pred_te = lr.predict(X_tr), lr.predict(X_te)

def adj_r2(r2, n_, k_):
    return 1 - (1 - r2) * (n_ - 1) / (n_ - k_ - 1)

k = X_final.shape[1]
perf = pd.DataFrame({
    'Train': [r2_score(y_tr, pred_tr), adj_r2(r2_score(y_tr, pred_tr), len(y_tr), k),
              mean_squared_error(y_tr, pred_tr), np.sqrt(mean_squared_error(y_tr, pred_tr)), mean_absolute_error(y_tr, pred_tr)],
    'Test':  [r2_score(y_te, pred_te), adj_r2(r2_score(y_te, pred_te), len(y_te), k),
              mean_squared_error(y_te, pred_te), np.sqrt(mean_squared_error(y_te, pred_te)), mean_absolute_error(y_te, pred_te)]},
    index=['R-squared', 'Adj R-squared', 'MSE', 'RMSE', 'MAE'])
display(perf)

cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)
cv_r2 = cross_val_score(LinearRegression(), X_final, y, cv=cv, scoring='r2')
print(f'5-fold CV R-squared: {cv_r2.mean():.4f} (+/- {cv_r2.std():.4f})')
gap = perf.loc['R-squared', 'Train'] - perf.loc['R-squared', 'Test']
print('Train R2 - Test R2 =', round(gap, 4), '-> ' + ('possible overfitting' if gap > 0.1 else 'no sign of overfitting'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_te, pred_te, alpha=0.6)
lims = [min(y_te.min(), pred_te.min()), max(y_te.max(), pred_te.max())]
axes[0].plot(lims, lims, 'r--'); axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted'); axes[0].set_title('Test set: Actual vs Predicted')
axes[1].scatter(pred_te, y_te - pred_te, alpha=0.6); axes[1].axhline(0, color='red', ls='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual'); axes[1].set_title('Test set residuals')
plt.tight_layout(); plt.show()

## Step 10: Save Results

In [ ]:
m['coef'].to_csv('regression_coefficients.csv')
vif_table.to_csv('vif_table.csv')
diag.sort_values('cooks_d', ascending=False).to_csv('influence_diagnostics.csv')
print('Saved in', os.getcwd(), ': regression_coefficients.csv, vif_table.csv, influence_diagnostics.csv')

## Theory Cheat-Sheet (for viva / written answers)

**Assumptions of linear regression (L.I.N.E. + no multicollinearity):**

| Assumption | How to check | If violated |
|---|---|---|
| **L**inearity of y and X | Residuals vs Fitted (no curve) | Add polynomial terms, transform variables (log, sqrt) |
| **I**ndependence of errors | Durbin-Watson (~2) | Time-series models, add lag terms |
| **N**ormality of errors | Q-Q plot, Shapiro-Wilk | Transform y (log/Box-Cox); less critical for large n |
| **E**qual variance (homoscedasticity) | Scale-Location plot, Breusch-Pagan | Transform y, weighted least squares, robust standard errors |
| No multicollinearity | VIF < 5 (or 10) | Drop/combine correlated predictors, ridge regression, PCA |

**Key formulas:** $R^2 = 1 - SSE/SST$ | Adjusted $R^2 = 1-(1-R^2)\frac{n-1}{n-p}$ | $RMSE=\sqrt{SSE/(n-p)}$ | $t_j = \hat\beta_j / SE(\hat\beta_j)$ | $F = \dfrac{SSR/(p-1)}{SSE/(n-p)}$

**Interpretation points**
* A coefficient = average change in y for a 1-unit increase in that predictor, **holding the others constant**.
* t-test p-value per coefficient (H0: coefficient = 0); F-test p-value for the whole model (H0: all slopes = 0).
* **Adjusted R-squared** penalises useless predictors; plain R-squared never decreases when a variable is added.
* **VIF** = 1/(1-R_j^2): how much the variance of a coefficient is inflated by correlation with other predictors.
* **Outlier** = unusual y (large residual); **high leverage** = unusual X; **influential** = removing it changes the fit a lot (large Cook's D). A point can be an outlier without being influential.
* Compare train vs test R-squared to detect overfitting; use cross-validation for a stable estimate.